# Tariff design under de-globalisation — SSR-MCDA replication notebook

**Saptadeep Biswas**, Department of Operations and Supply Chain, GITAM School of Business, GITAM (Deemed to be University) — `sbiswas3@gitam.edu`

This notebook reproduces the analysis of the paper end to end, layer by layer, on the authentic data shipped in `../data/`.

| Layer | What it does | Module |
|---|---|---|
| I | Generates the decision matrix as the unique subgame-perfect equilibrium of a Stackelberg tariff game | `ssr_mcda.game` |
| II | Identifies a 2-additive Choquet capacity from a cooperative *discrimination game* on the criteria | `ssr_mcda.capacity` |
| III | Ranks by minimax coalitional regret and certifies a stability radius, both exact LPs | `ssr_mcda.regret` |

Run the cells in order. The full pipeline (excluding the 2 000-draw Monte Carlo) takes about a minute.

In [ ]:
import os, sys, json
import numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath('.'))

import ssr_mcda as S
from ssr_mcda import benchmarks as B
from ssr_mcda.data import build_case, labour_intensities, preference_indices, REVEALED_PREFERENCES
from ssr_mcda.regret import UncertaintySet, mcr_ranking, preference_rows, stability_radius, regret_attribution

pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)
np.set_printoptions(suppress=True, precision=4)

CASE = 'USA'          # switch to 'IND' to reproduce the India case
POST, CRIT = S.POSTURE_CODES, S.CRITERION_CODES
print('ssr_mcda', S.__version__)

## 0. Verify the formal claims first

Every proposition and theorem in the paper has an executable test named after it. Nothing below should be trusted if these do not pass.

In [ ]:
import subprocess
print(subprocess.run([sys.executable, 'tests/test_ssr_mcda.py'], capture_output=True, text=True).stdout[-900:])

## 1. Load the authentic inputs

Applied tariffs come from UNCTAD-TRAINS via the World Bank WITS API; sourcing shares from WITS TradeStats; the Logistics Performance Index from the World Bank WDI; geopolitical alignment from the UNGA ideal-point scores published by Airaudo et al. (2025).

`build_case` raises rather than imputes if any input is missing — nothing in this pipeline is filled in silently.

In [ ]:
par  = S.GameParams()
case = build_case(CASE, par)

env = pd.DataFrame({'origin': case.env.origins,
                    'geopolitical_distance': case.env.alignment.round(3),
                    'distance_km': case.env.distance_km.round(0),
                    'LPI': case.env.lpi})
display(env)

display(pd.DataFrame([{'sector': s.name, 'HS6': '+'.join(s.hs6),
                       'applied_MFN_%': round(100*s.tau0[1], 3),
                       'import_value_USDk': f"{s.import_value:,.0f}"} for s in case.sectors]))

## 2. Layer I — the decision matrix as an equilibrium object

Unit costs are *recovered* from the observed sourcing shares by inverting the Stage-3 KKT conditions (Proposition 3). We check first that this reproduces the observed allocation exactly — the counterfactual must start from the world as observed.

In [ ]:
from ssr_mcda.game import calibrate_costs, sourcing_best_response

for s in case.sectors:
    c = calibrate_costs(s.x0, s.tau0, case.env.alignment, par.rho, par.lam)
    x = sourcing_best_response(c*(1+s.tau0) + par.lam*case.env.alignment, par.rho)
    print(f"{s.name:16s}  max|x* - x0| = {np.abs(x - s.x0).max():.2e}")

In [ ]:
P, costs, detail = S.build_performance_matrix(
    case.sectors, case.env, par, case.rival, labour_intensities())
Z = S.normalise(P)

print('Raw equilibrium performance matrix P:')
display(pd.DataFrame(P, index=POST, columns=CRIT).round(4))
print('\nNormalised, benefit-oriented Z:')
display(pd.DataFrame(Z, index=POST, columns=CRIT).round(3))

In [ ]:
# The mechanism behind the numbers: how each posture reallocates sourcing
sec = 'Semiconductors'
shares = pd.DataFrame({code: d[sec]['x'] for code, d in detail.items()},
                      index=case.env.origins)
display(shares.round(3))

## 3. Layer II — criterion weights as the solution of a cooperative game

The criteria are the *players*. A coalition's worth is the share of documented policy choices it can reproduce on its own. The preference set comes from recorded policy actions, not a questionnaire.

In [ ]:
for a, b, why in REVEALED_PREFERENCES[CASE]:
    print(f"  {a} >= {b}   {why}")
prefs = preference_indices(CASE, POST)
print(f"\n{len(prefs)} statements out of {len(POST)*(len(POST)-1)} ordered pairs "
      f"-> the rest of the ranking is out-of-sample")

In [ ]:
cap, info = S.cci(Z, prefs, verbose=True)

print('\nIs the projection a genuine capacity?', cap.is_capacity(),
      f"(max monotonicity violation {cap.monotonicity_violation():.2e})")
display(pd.DataFrame({'shapley_of_game': info['shapley_of_game'].round(3),
                      'importance_of_capacity': cap.shapley().round(3),
                      'moebius_singleton': cap.singles.round(3)}, index=CRIT))
print(f"orness = {cap.orness():.3f}   (0.5 = additive)      "
      f"normalised entropy = {cap.entropy():.3f}")

In [ ]:
# Interaction indices: negative = redundant criteria, positive = complementary
I = pd.DataFrame(cap.interaction_matrix(), index=CRIT, columns=CRIT).round(3)
display(I.style.background_gradient(cmap='RdBu', vmin=-abs(I.values).max(),
                                    vmax=abs(I.values).max()))

## 4. Layer III — minimax coalitional regret and the stability radius

The Choquet integral is linear in the Möbius coefficients, so worst-case regret over the capacity uncertainty set is an exact linear programme — not a simulation.

In [ ]:
G = preference_rows(Z, prefs)
U = UncertaintySet(n=len(CRIT), centre=cap.vector, epsilon=0.05, preference_rows=G)
res = mcr_ranking(Z, U)

out = pd.DataFrame({'choquet_score': np.asarray(cap.choquet(Z)).round(4),
                    'max_regret': res['regret'].round(4),
                    'mcr_rank': res['rank'],
                    'binding_adversary': [POST[i] if i >= 0 else '' for i in res['adversary']]},
                   index=POST)
display(out.sort_values('mcr_rank'))
print('Recommended posture:', POST[res['recommended']])

In [ ]:
for norm in ('l1', 'linf'):
    rad, chal, _ = stability_radius(Z, cap.vector, res['recommended'], norm=norm)
    print(f"stability radius ({norm:4s}) = {rad:.4f}   "
          f"nearest challenger: {POST[chal]}")
print('\nInterpretation: no capacity within this distance of the identified one'
      '\noverturns the recommendation, and there exists one at exactly that distance that does.')

In [ ]:
# Where does each posture's regret come from?
rows = []
for i, p in enumerate(POST):
    b = res['adversary'][i]
    if b < 0:
        continue
    rows.append({'posture': p, 'adversary': POST[b],
                 **{k: round(v, 4) for k, v in regret_attribution(Z, i, b, U, CRIT).items()}})
display(pd.DataFrame(rows).set_index('posture'))

## 5. How far can the capacity be wrong?

Sweeping the uncertainty radius traces the robustness frontier. The recommendation holds zero regret up to the certified radius.

In [ ]:
from ssr_mcda.regret import max_regret
rows = []
for eps in [0.0, 0.02, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30]:
    Ue = UncertaintySet(n=len(CRIT), centre=cap.vector, epsilon=eps, preference_rows=G)
    Rv, _ = max_regret(Z, Ue)
    rows.append({'epsilon': eps, 'recommended': POST[int(np.argmin(Rv))],
                 **{p: round(Rv[i], 4) for i, p in enumerate(POST)}})
display(pd.DataFrame(rows).set_index('epsilon'))

## 6. Comparison with established procedures

Nine comparators, all fed the same normalised matrix so the comparison isolates the weighting and aggregation logic.

In [ ]:
phi = cap.shapley()
methods = {
    'SSR (Choquet)':      (np.asarray(cap.choquet(Z)), True),
    'SSR (minimax regret)': (res['regret'], False),
    'WSM (Shapley)':      (B.weighted_sum(Z, phi), True),
    'WSM (entropy)':      (B.weighted_sum(Z, B.entropy_weights(Z)), True),
    'WSM (CRITIC)':       (B.weighted_sum(Z, B.critic_weights(Z)), True),
    'TOPSIS':             (B.topsis(Z, phi), True),
    'VIKOR (Q)':          (B.vikor(Z, phi)['Q'], False),
    'PROMETHEE II':       (B.promethee2(Z, phi), True),
    'COPRAS':             (B.copras(Z, phi), True),
    'SMAA-2 (rank-1)':    (B.smaa2(Z, n_draws=20000)['acceptability'][:, 0], True),
}
ranks = pd.DataFrame({m: B.scores_to_ranks(s, hib) for m, (s, hib) in methods.items()},
                     index=POST)
display(ranks)

base = ranks['SSR (minimax regret)']
display(pd.DataFrame({m: B.rank_agreement(base, ranks[m]) for m in ranks.columns}).T.round(3))

## 7. Does the conclusion survive the parameters being wrong?

A short Monte Carlo over the eleven behavioural parameters, including the full declared range of the one calibration parameter. Increase `DRAWS` to 2000 to reproduce the figure in the paper (about nine minutes).

In [ ]:
from run_case_study import monte_carlo
DRAWS = 200
mc = monte_carlo(CASE, par, DRAWS)
acc = pd.DataFrame({f'rank {r}': [(mc[f'rank_{p}'] == r).mean() for p in POST]
                    for r in range(1, len(POST)+1)}, index=POST)
display(acc.round(3))
print(f"\nrank-1 acceptability of the recommendation "
      f"({POST[res['recommended']]}): "
      f"{100*(mc['rank_' + POST[res['recommended']]] == 1).mean():.1f}% of {len(mc)} draws")

## 8. Reproduce the full study

The cells above cover one economy at reduced Monte-Carlo resolution. The commands below regenerate every table, number and figure in the manuscript for both economies.

```bash
python run_case_study.py --draws 2000   # all result tables
python make_numbers.py                  # numbers.tex + tables/*.tex
python make_figures.py                  # all 13 figures as vector PDF
python make_seps.py                     # the SEPS manuscript variant
```